
## 1. What is Structured Streaming?


- **Structured Streaming** is a `Spark engine` used to `process continuously arriving data in real time`.
- It processes `incoming data continously` using the `DataFrame and SQL APIs`.


### Real-time example:


**Consider an e-commerce application like Amazon.**

1. Customers place orders continuously.
2. Orders arrive from Kafka.
3. Spark reads incoming orders.
4. Invalid orders are filtered.
5. Valid orders are processed.
6. Sales data is stored in Delta Lake.


## Important Interview Questions for Structured Streaming:


**Q1. What is the default processing model in Structured Streaming?**

`Micro-batch processing.`


**Q2. Is Structured Streaming fault tolerant?**

- **Yes**. 
- It uses `checkpointing and recovery` mechanisms to `resume processing after failures`.


**Q3. Can we use DataFrame transformations in Structured Streaming?**

- **Yes**. 
- Operations such as `select(), filter(), withColumn()` and `supported aggregations` can be used.


**Q4. What is an unbounded table?**

An **unbounded table** represents `continuously arriving data` that keeps `growing` over time.


**Q5. Does Structured Streaming always process data continuously?**

- **No**. 
- It supports **continuous-running queries as well as trigger-based incremental processing**, such as `ProcessingTime and AvailableNow`.


**Q6. What is the difference between Structured Streaming and DStreams?**

| DStreams | Structured Streaming |
|---|---|
| Based on RDDs | Based on DataFrames and SQL |
| Uses micro-batch processing | Uses micro-batch by default |
| Lower-level programming model | Higher-level declarative programming model |
| Legacy Spark Streaming API | Modern Spark streaming API |


## 2. What are Sources and Sinks?


**Source:**

 A **source** is where `streaming data comes from`.

**Sink:** 

A **sink** is where `processed streaming data is stored`.


| Sources | Sinks |
|---|---|
| Kafka | Delta Lake |
| Files (CSV, JSON, Parquet) | Console |
| Auto Loader | Kafka |
| Rate Source | Memory |
| Socket | Files |


## Real-time example:


**An e-commerce application receives orders from Kafka and stores processed orders in Delta Lake.**



```text
+-----------------------------+
|       SOURCE: KAFKA         |
|                             |
|   Incoming Customer Orders  |
+-----------------------------+
              |
              v
+-----------------------------+
| SPARK STRUCTURED STREAMING  |
|                             |
|  1. Filter Invalid Records  |
|  2. Clean Data              |
|  3. Transform Data           |
+-----------------------------+
              |
              v
+-----------------------------+
|       SINK: DELTA LAKE      |
|                             |
|   Store Processed Orders    |
+-----------------------------+


## Important Interview Questions for Source and Sink:


**Q1. Can Kafka act as both a Source and a Sink?**

- **Yes**. 
- Spark can read `events from one Kafka topic`, process them and publish the results to `another Kafka topic`.


**Q2. What is the difference between a Source and a Sink?**

**Source:**

 A **source** is where `streaming data comes from`.

**Sink:** 

A **sink** is where `processed streaming data is stored`.


**Q3. Can we use multiple sources in Structured Streaming?**

- **Yes**. 
- We can read from `multiple streaming sources` and combine their `DataFrames` using **supported operations**.


**Q4. Is Console a production sink?**

- **No**.
- **Console** is used for `debugging and testing`, because it displays `results` rather than providing` durable production storage`.


**Q5. What happens if a Sink fails?**

- The **streaming query** may fail, 
- With a `properly configured checkpoint and a compatible sink`, Spark can `recover and resume` processing.


## 3. What is a Trigger in Structured Streaming?


A **trigger** in Structured Streaming `controls when Spark processes incoming streaming data`.



### Example:


- **ProcessingTime** with a 10-second interval processes data every 10 seconds. 
- Other trigger types include **Once, AvailableNow, Default and Continuous**. 
- We choose a trigger based on **latency, requirements and cost**.


## Types of Triggers


| Trigger Type | Explanation |
|---|---|
| Default Trigger | Processes data as soon as possible using micro-batches. |
| ProcessingTime | Processes data at a specified time interval. |
| Once | Processes all data in a single batch. |
| AvailableNow | Processes available data in multiple batches. |
| Continuous | Processes data continuously with low latency (experimental). |


### Real-time example:


**A banking application processes transactions every 10 seconds.**

In [0]:
query = (
    orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start(output_path)
)


## Important Interview Questions for Tiggers:


**Q1. What happens if a micro-batch takes longer than the configured trigger interval?**

- `Spark does not run overlapping micro-batches` for the same streaming query. 
- The **next batch starts** after the `current batch finishes`, potentially `causing a delay`.


**Q2. Does a trigger control data arrival frequency?**

- **No**.
- A **trigger** in Structured Streaming `controls when Spark processes incoming streaming data`.


**Q3. What is the difference between ProcessingTime and AvailableNow?**

- **ProcessingTime:** Keeps `processing data` at `configured intervals`.
- **AvailableNow:** Processes `currently available data and stops`.


**Q4. Does AvailableNow process only one micro-batch?**

- **No**. 
- It can process `available data through multiple micro-batches` before stopping.


**Q5. Can we change the trigger interval?**

- **Yes**. 
- We can **configure the trigger interval** when `defining the streaming query`.


### 4. What are Output Modes?


**Output Modes** determine `how the results of a streaming query are stored to the output sink`.


### Types of Output Modes:


**1. Append Mode**

- **Append Mode** `writes` the `added rows to the output sink`. 
- Once a row is written, it is not updated again.
- Suitable for new customer orders.

**2. Complete Mode**

- **Complete Mode** `writes` the `complete table rows to the output sink` after every trigger.
- Suitable for total sales by region.

**3. Update Mode**

- **Update Mode** `writes` the `rows whose values have changed`.
- Unchanged rows are not written again.
- Suitable for continuously updated sales totals.


### Real-Time Example:


**Suppose we calculate total sales by region.**

| Region | Total Sales |
|---|---:|
| Chennai | 50,000 |
| Bangalore | 75,000 |


## Important Interview Questions for Output Modes:


**Q1. How many output modes are available in Structured Streaming?**

**Three**: `Append, Complete,` and `Update`.


**Q2. Which output mode writes the entire result table?**

`Complete Mode`.


**Q3. Which output mode writes only newly added records?**

`Append Mode`.


**Q4. Which output mode writes only modified records?**

`Update Mode`.


**Q5. Can we use all three output modes with every sink?**

- **No**. 
- `Supported output modes` depend on the **sink**. 
- **For example:** The `console sink` supports all **three output modes**, while other sinks may have `restrictions`.


## 5. What is a Watermark?


- A **Watermark** in **Structured Streaming** determines `how long Spark should wait for late-arriving data`,
- Before considering an `aggregation window complete and cleaning up old state`.


**An order is created at 10:00 AM, but due to network delay, it reaches Spark at 10:15 AM.**

In [0]:
df = df.withWatermark("order_time", "10 minutes")


## Why do we use watermarks?


- Handle late-arriving data.
- Control state-store size.
- Prevent unlimited state growth.
- Support event-time window aggregations.


### How to implement Watermark in PySpark?


**Example:**

**Calculate total sales every 5 minutes.**

In [0]:
from pyspark.sql.functions import window, sum

result_df = orders_df \
    .withWatermark("order_time", "10 minutes") \
    .groupBy(
        window("order_time", "5 minutes")
    ) \
    .agg(
        sum("amount").alias("total_sales")
    )

query = result_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()


**Explanation:**

1. **withWatermark()** defines the allowed lateness threshold.
2. **order_time** is the event-time column.
3. **10 minutes** defines the watermark delay threshold.
4. **groupBy(window())** creates 5-minute event-time windows.
5. **sum()** calculates total sales for each window.
6.** Append mode** outputs finalized aggregation results.


### Watermark in DataFrame API vs SQL:


**Watermark in DataFrame API:**

In [0]:
df.withWatermark("event_time", "10 minutes")


**Spark SQL:**

In [0]:
%sql
SELECT
    window(event_time, '5 minutes') AS time_window,
    SUM(amount) AS total_sales
FROM STREAM orders
WATERMARK event_time DELAY OF INTERVAL 10 MINUTES
GROUP BY window(event_time, '5 minutes');


## Important Interview Questions for watermark:


**Q1. What is the main purpose of a Watermark?**

- A **Watermark** in Structured Streaming determines `how long Spark should wait for late-arriving data`,
- Before considering an `aggregation window complete and cleaning up old state`.


**Q2. Is Watermark based on Event Time or Processing Time?**

Watermark is based on **Event Time**.

- **Event Time:** When the event actually `occurred`.
- **Processing Time:** When Spark `processes` the event.


**Q3. Is Watermark mandatory in Structured Streaming?**

- **No**. 
- It is `optional`, but `useful for handling late data` and `managing state in event-time aggregations`.


**Q4. What happens if data arrives after the Watermark threshold?**

- `data arriving too late` may be dropped if its window state has `already been cleaned up`. 
- **Exact behavior** depends on the `query and output mode`.


**Q5. Does Watermark remove duplicate records?**

- **No**. 
- Watermark **does not automatically remove duplicates**. 
- It can help bound the state used by `streaming deduplication`.


**Q6. What happens if we don't use Watermark?**

Spark may `retain aggregation state indefinitely` for certain streaming queries, causing `state growth` and increased `resource consumption`.


**Q7. Can we use Watermark with Sliding Windows?**

- **Yes**. 
- **Watermarks** can be used with `sliding event-time window aggregations`.


## 6. What are Windows in Structured Streaming?


**Windows** is a technique used to `group streaming records based on a time interval and perform aggregations` such as `count, sum, and average`.


### Types of Windows:


| Window Type | Explanation | Real-Time Example |
|---|---|---|
| Tumbling Window | A Tumbling Window divides streaming data into fixed-size, non-overlapping time intervals. | Calculate sales every 5 minutes. |
| Sliding Window | A Sliding Window divides streaming data into fixed-size time intervals. | Calculate sales for the last 10 minutes, updating every 5 minutes. |
| Session Window | It creates a window when session starts and closes it after a specified inactivity gap. | Group customer clicks until the customer is inactive for 10 minutes. |


### Real-Time Example:


**An e-commerce company wants to calculate total sales every 5 minutes.**

| Time Window | Total Sales |
|---|---:|
| 10:00–10:05 | ₹50,000 |
| 10:05–10:10 | ₹75,000 |
| 10:10–10:15 | ₹90,000 |


### Example – Tumbling Window:

In [0]:
from pyspark.sql.functions import *

result = df \
    .withWatermark("order_time", "10 minutes") \
    .groupBy(
        window("order_time", "5 minutes")
    ) \
    .agg(
        sum("amount").alias("total_sales")
    )


## Important Interview Questions for Windows:


**Q1. How many main types of time windows are available in Structured Streaming?**

**Three**: `Tumbling, Sliding,` and `Session` Windows.


**Q2. What is the difference between Tumbling and Sliding Windows?**

**A. Tumbling Window:**

 A Tumbling Window divides `streaming data` into `fixed-size`,` non-overlapping time intervals`. 

**B. Sliding Window:**

 A Sliding Window divides `streaming data` into `fixed-size time intervals`. 


**Q3. Can one event belong to multiple Sliding Windows?**

- **Yes**. 
- An event can belong to `multiple windows` because the `windows overlap`.


**Q4. What is a Session Window?**

A **Session Window** is used to `group events based on user activity` and `ends after a specified inactivity gap`.


**Q5. What is the difference between window duration and slide duration?**

- **Window duration:** How long `each window covers`.
- **Slide duration:** How `frequently a new window` starts.

**Example**: window("order_time", "10 minutes", "5 minutes").

- **Window duration** = 10 minutes.
- **Slide duration** = 5 minutes.


**Q6. Is a Watermark required for every window?**

- **No**.
- **Watermarks** are commonly used with `event-time windows` to handle `late-arriving data` and control `state retention`.


## 7. What is Checkpointing?


- **Checkpointing** is a `fault-tolerance` mechanism used to stores `streaming query progress` and `state information` in storage. 
- If a failure occurs, Spark can `recover and continue processing` using the checkpointing.


### Real-time example:


**An e-commerce streaming application processes 10,000 orders. Suddenly, the Spark cluster crashes.**

**After restarting the query with the same checkpoint location, Spark can recover its progress and state instead of starting from scratch.**

In [0]:
query = df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/checkpoints/orders"
    ) \
    .start("/delta/orders")


## Why do we use checkpointing?


- Recover from failures.
- Maintain streaming state.
- Support fault-tolerant processing.


## Important Interview Questions for Checkpointing:


**Q1. What is the main purpose of Checkpointing?**

- **Checkpointing** is a `fault-tolerance` mechanism used to stores `streaming query progress` and `state information` in storage. 
- If a failure occurs, Spark can `recover and continue processing` using the checkpointing.


**Q2. Is Checkpointing mandatory in Structured Streaming?**

- **Yes**, 
- for `fault-tolerant` production streaming, **checkpointing** is essential. 
- However, it is **not mandatory** in every `Structured Streaming query configuration`.


**Q3. Can we use the same checkpoint location for multiple streaming queries?**

- **No**. 
- `Each streaming query` should have its `own checkpoint location`.


**Q4. Can we delete a checkpoint directory?**

- **No**,
- You `should not delete` a checkpoint directory. 
- Deleting it can `cause the query to lose` its `recovery information` and potentially `reprocess data`.


**Q5. Does Checkpointing guarantee exactly-once processing?**

- **No**. 
- **Checkpointing** alone **does not guarantee** for `end-to-end exactly-once` processing. 
- The **source and sink** must also support the `necessary recovery` and `transactional guarantees`.


**Q6. Where should we store checkpoint files?**

Use reliable storage such as:

- Amazon S3
- Azure Data Lake Storage (ADLS)
- Google Cloud Storage (GCS)
- Supported distributed storage.


## 8. What is Kafka Integration with Structured Streaming?


- **Kafka integration** is a **Spark Structured Streaming** used to `consume real-time messages from Kafka topics`,
- `process them` using Spark **transformations**, 
- `write the results to destinations` such as **Delta Lake**.


### Real-time example:


**A banking application publishes transaction messages to a Kafka topic. Spark reads those transactions, identifies suspicious transactions, and stores the processed records in Delta Lake for further analysis.**


![image_1793702368798.png](./image_1793702368798.png "image_1793702368798.png")


### PySpark – Read from Kafka:

In [0]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    ) \
    .option("subscribe", "transactions") \
    .load()


### Convert Kafka value into readable text:

In [0]:
from pyspark.sql.functions import *

orders_df = kafka_df.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("timestamp")
)


### Write to Delta Lake:

In [0]:
query = orders_df.writeStream \
    .format("delta") \
    .option(
        "checkpointLocation",
        "/checkpoints/transactions"
    ) \
    .start("/delta/transactions")



## Important Kafka Columns in Spark Structured Streaming:


| Column | Explanation | Example |
|---|---|---|
| key | Message key used to identify or partition messages. | customer_101 |
| value | Actual message data (payload). | {"customer_id":101,"amount":5000} |
| topic | Kafka topic name from which the message is received. | customer_orders |
| partition | Partition number where the message is stored. | 0 |
| offset | Unique sequential position of a message within a partition. | 125 |
| timestamp | Timestamp associated with the Kafka message. | 2026-11-03 10:30:00 |



### Real-Time Example:


**An e-commerce company sends customer orders to Kafka.**

| Column | Value |
|---|---|
| key | customer_101 |
| value | {"order_id":501,"amount":5000} |
| topic | customer_orders |
| partition | 0 |
| offset | 125 |
| timestamp | 2026-11-03 10:30:00 |


## Important Interview Questions for Kafka:


**Q1. What is the role of Kafka in Structured Streaming?**

- **Kafka integration** is a **Spark Structured Streaming** used to `consume real-time messages from Kafka topics`,
- `process them` using Spark **transformations**, 
- `write the results to destinations` such as **Delta Lake**.


**Q2. Can Spark Structured Streaming read multiple Kafka topics?**

- **Yes**,
- We can use the `subscribe option` with `comma-separated topic names`.
 
- **.option("subscribe", "orders,payments,customers")**


**Q3. What happens if Spark stops unexpectedly?**

Spark can recover **streaming progres**s using `checkpoint information` and `resume processing` from the **recorded offsets**, subject to the configured source and sink guarantees.


**Q4. Does Kafka store data permanently?**

- **Not necessarily.**
- Kafka `retains` data according to `configured retention policies`, based on `time or storage size`.


**Q5. What is the difference between Kafka and Spark Structured Streaming?**

| Apache Kafka | Spark Structured Streaming |
|---|---|
| Distributed messaging platform | Distributed stream processing engine |
| Collects and stores events | Processes incoming events |
| Uses topics and partitions | Uses DataFrames and transformations |
| Distributes messages between producers and consumers | Performs filtering, joins and aggregations |
| Acts as a streaming data source or message broker | Acts as a streaming data processing engine |
| Supports message retention and replay | Supports incremental processing and fault tolerance |
| Uses producers and consumers | Uses streaming queries and sinks |
| Example: Receives customer order events | Example: Calculates real-time sales from customer orders |


### 9. What is Auto Loader in Databricks?


**Auto Loader** is a `Databricks feature` used to `automatically load new files from cloud storage into Delta Lake using Spark Structured Streaming`.


**Supported file formats:**

- CSV
- JSON
- Parquet
- Avro
- ORC
- Text
- Binary files


### Real-time example:


**Consider an e-commerce application where customer order files arrive continuously in cloud storage.**

1. New customer order files arrive in AWS S3 or Azure Data Lake.
2. Auto Loader detects newly arriving files.
3. Spark reads and processes the files.
4. Data is cleaned and transformed.
5. Processed records are stored in a Delta Lake table.


### Auto Loader + Structured Streaming Architecture



```text
┌──────────────────────────────────────────────────────────────┐
│                    1. CLOUD STORAGE (SOURCE)                 │
│                                                              │
│       AWS S3  /  Azure Data Lake  /  Google Cloud Storage    │
│                                                              |
|       - where streaming data comes from.                     │
│       • Incoming files like CSV, JSON and Parquet files.     │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                    2. DATABRICKS AUTO LOADER                 │
│                                                              │
│       • Automatically loads new files from cloud storage     |
|           to delta lake                                      │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                 3. SPARK STRUCTURED STREAMING                │
│                                                              │
│       • Filters unwanted records                             │
│       • Cleans missing and duplicate data                    │
│        • Applies business logic                              │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                       4. DELTA LAKE (SINK)                   │
│                                                              │
│       • Stores processed customer orders                     │
│       • Enables reliable dashboard and analytical queries    │
└──────────────────────────────────────────────────────────────┘



### Data Flow Summary:



| Component | Responsibility |
|---|---|
| Cloud Storage | Stores incoming files |
| Auto Loader | Detects and incrementally loads new files |
| Spark Structured Streaming | Processes and transforms data |
| Delta Lake | Stores processed data |




### Important Auto Loader Options:



| Option | Purpose |
|---|---|
| cloudFiles | Enables Auto Loader |
| cloudFiles.format | Specifies source file format |
| cloudFiles.schemaLocation | Stores inferred schema information |
| checkpointLocation | Tracks streaming progress and processed files |

In [0]:
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation",
            "/Volumes/workspace/default/data_volume/schema")
    .load("/Volumes/workspace/default/data_volume/orders")
)

(df.writeStream
    .format("delta")
    .option("checkpointLocation",
            "/Volumes/workspace/default/data_volume/checkpoint")
    .outputMode("append")
    .start("/Volumes/workspace/default/data_volume/processed_orders")
)


## Important Interview Questions for Auto Loader:


**Q1. Is Auto Loader a batch or streaming feature?**

**Auto Loader** is a `Databricks feature` used to `automatically load new files from cloud storage into Delta Lake using Spark Structured Streaming`.


**Q2. What is the difference between schemaLocation and checkpointLocation?**

- **schemaLocation** stores `schema information` and `schema evolution history`.
- **checkpointLocation** stores `streaming progress` and `file-processing state`.


**Q3. Does Auto Loader process the same file repeatedly?**

- Under normal operation, Auto Loader tracks **discovered and processed files** using `checkpoint state`, avoids `reprocessing them`. 
- `Exactly-once processing `of the complete pipeline also depends on the sink and its write semantics.


**Q4. Which file formats does Auto Loader support?**

`JSON, CSV, Parquet, text` and `binary files.`


**Q5. What are the two file discovery modes in Auto Loader?**

1. Directory listing mode.
2. File notification mode.


## 10. What are Streaming Tables?


- **Streaming Tables** are `Delta tables` in Databricks used to `store and process data continuously`.
- They `continuously ingest new records` as they arrive, without reprocessing the entire dataset.
- They are commonly used in `Lakeflow Spark Declarative Pipelines` to build real-time data pipelines.


### Real-time example:


**Consider an e-commerce application receiving customer orders continuously.**

1. Customer places an order.
2. Order data arrives from Kafka.
3. Streaming Table processes incoming orders incrementally.
4. Invalid records are filtered.
5. Valid orders are stored in a Streaming Table.
6. Downstream analytics can consume the processed data.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE orders_stream
AS
SELECT *
FROM STREAM read_files(
    '/mnt/orders/',
    format => 'csv',
    header => true
);


## Important Interview Questions for Streaming Tables:


**Q1. Are Streaming Tables stored in Delta format?**

- **Yes**. 
- **Databricks streaming tables** use `Delta Lake storage`.


**Q2. Can we use Kafka as a source for Streaming Tables?**

- **Yes.**
- We can consume `Kafka events` using `Spark Structured Streaming` and use the resulting `streaming DataFrame to define a streaming table`.


**Q3. Can Streaming Tables handle schema evolution?**

- **Yes**. 
- They support` schema evolution depending on the source`, `pipeline configuration` and `schema handling settings`.


**Q4. Where do we use Streaming Tables in the Medallion Architecture?**

Primarily in `Bronze and Silver layers` for incremental `ingestion` and `transformation`.


**Q5. Are Streaming Tables and Structured Streaming the same?**

**No**.
- **Structured Streaming** is `Spark's engine`.
- **Streaming Tables** are `Databricks-managed tables` built using declarative pipeline functionality.


## 11. What are Materialized Views?


- **Materialized Views** are `database objects` used to `store the precomputed results` of a SQL query.
- Instead of executing the query every time, Databricks maintains` the query results and refreshes them`.
- They are mainly used to `improve query performance` and `simplify reporting and analytics`.


### Real-time example:


**Consider an e-commerce company with millions of customer orders.**

**You need to calculate total sales for each region.**

**Without a Materialized View:**

- Every time a user runs the query, Spark processes the source data again.
- This can take more time for large datasets.

**With a Materialized View:**

- Databricks maintains the calculated sales results.
- Users can query the stored results instead of recalculating everything from scratch.


### SQL Example:


**Suppose we have an orders table with region and amount columns.**

In [0]:
%sql
CREATE MATERIALIZED VIEW regional_sales AS
SELECT
    region,
    SUM(amount) AS total_sales,
    COUNT(*) AS total_orders
FROM orders
GROUP BY region;


### To query the Materialized View:

In [0]:
%sql
SELECT *
FROM regional_sales;


## Difference Between Streaming Tables and Materialized Views


| Streaming Tables | Materialized Views |
|---|---|
| Designed for incremental data ingestion and processing. | Designed to store the results of a query. |
| Commonly processes new incoming records. | Maintains the results of a SQL query. |
| Useful for ingestion and streaming pipelines. | Useful for analytical queries and reporting. |
| Uses streaming semantics. | Supports incremental or full refresh. |
| Processes data incrementally as new data arrives. | Refreshes query results when source data changes. |
| Example: New customer orders ingestion. | Example: Total sales by region. |


## Important Interview Questions for Materialized View:


**Q1. Does a Materialized View store data?**

- **Yes**. 
- **Materialized View** stores the `results of its defining query`.


**Q2. Does a Materialized View automatically update when source data changes?**

- **No**, 
- A materialized view stores the `results of a query`. 
- **When the source table changes**, the materialized view must be `refreshed or incrementally updated` to reflect the latest data.


**Q3. Does a Materialized View always perform incremental refresh?**

- **No**. 
- Databricks uses `incremental refresh` when the `query and source operations` support it. 
- Otherwise, it may perform a `full refresh`.


**Q4. Why do we use Materialized Views?**

To `reduce repeated query computation`, `improve reporting performance` and `simplify access to aggregated data`.



**Q5. Can we create a Materialized View using GROUP BY?**

- **Yes**. 
- **Aggregations** such as `SUM(), COUNT() and AVG()` are common use cases, subject to supported query definitions.


**Q6. Does a Materialized View always improve performance?**

- **No**. 
- It adds `storage` and `refresh costs`. 
- It is **beneficial** when` expensive queries are executed repeatedly` and the `stored results are reused`.


## 12. What is DLT?


- **DLT (Delta Live Tables)** is a `Databricks framework` used to `build, manage, and automate` reliable data pipelines.
- It simplifies `data ingestion, data quality, transformations, dependencies`.


### Real-time example:


**Consider an e-commerce company receiving customer orders from Kafka and cloud storage.**

DLT helps build an automated pipeline with three layers:
- **Bronze**: Store raw customer orders.
- **Silver**: Clean data, remove duplicates, and filter invalid records.
- **Gold**: Calculate total sales, revenue, and regional reports.


### DLT Architecture – Medallion Architecture:


```text
┌──────────────────────────────────────────────────────────────┐
│                  1. DATA SOURCES (SOURCE LAYER)              │
│                                                              │
│       Kafka / AWS S3 / Azure Data Lake / CSV / JSON          │
│                                                              │
│       • Receives raw customer order data                     │
│       • Supports batch and streaming ingestion               │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                    2. BRONZE LAYER (RAW DATA)                │
│                                                              │
│       • Stores raw customer order data                       │
│       • Maintains original source data                       │
│       • Preserves data for auditing and reprocessing         │
│                                                              │
│       Example: Raw customer orders                           │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                   3. SILVER LAYER (CLEANED DATA)             │
│                                                              │
│       • Cleans and validates records                         │
│       • Removes duplicate records                            │
│       • Handles missing and invalid values                   │
│       • Applies data quality checks                          │
│                                                              │
│       Example: Clean and validated customer orders           │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                   4. GOLD LAYER (BUSINESS DATA)              │
│                                                              │
│       • Performs sales aggregations                          │
│       • Calculates total revenue and profit                  │
│       • Creates business-level summary tables                │
│       • Prepares data for reporting and analytics            │
│                                                              │
│       Example: Sales by region and monthly revenue           │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                     5. REPORTING LAYER                       │
│                                                              │
│                  Power BI / SQL Dashboards                   │
│                                                              │
│       • Connects to Gold tables                              │
│       • Generates business reports                           │
│       • Displays KPIs and analytical dashboards              │
└──────────────────────────────────────────────────────────────┘


### Layer Comparison:


| Layer | Purpose | Example |
|---|---|---|
| Bronze | Store raw data | Raw customer orders |
| Silver | Clean and validate data | Deduplicated orders |
| Gold | Business-level aggregation | Total sales by region |
| BI / Reporting | Visualize business insights | Power BI dashboard |




### Real-Time E-Commerce Example:


| Layer | Data Processing |
|---|---|
| Bronze | Store 15,000 raw customer orders |
| Silver | Remove duplicates and invalid orders |
| Gold | Calculate daily sales and regional revenue |
| Power BI | Display sales performance dashboard |


## What are the two Pipeline Modes in DLT?


1. Triggered Pipeline Mode
2. Continuous Pipeline Mode


### 1. Triggered Pipeline Mode:


- **Triggered pipeline mode** executes the `pipeline using the data available` when` the pipeline update starts`.
- After `processing the available data` and `refreshing the tables`, the pipeline automatically stops.


### Real-time example:


**An e-commerce company receives customer order files every hour.**

- At 10:00 AM, the pipeline starts.
- It processes all available order files.
- It cleans and transforms the data.
- It updates Bronze, Silver and Gold tables.
- The pipeline stops after completing the update.


### B. Continuous Pipeline Mode:


**Continuous mode** keeps the pipeline `running and processes new data as it arrives` to maintain fresh tables.


### Real-time example:


**An e-commerce company receives customer orders continuously from Kafka.**

- Customer places an order.
- New order data arrives in Kafka.
- The pipeline processes incoming records.
- Bronze, Silver and Gold tables are updated.
- Processing continues as new data arrives.


## Important Interview Questions for DLT:


**Q1. What is the difference between DLT and Structured Streaming?**

- **Structured Streaming** is a `Spark engine` used to `process continuously arriving data in real time`.
- It processes `incoming data continously` using the `DataFrame and SQL APIs`.


- **DLT (Delta Live Tables)** is a `Databricks framework` used to `build, manage, and automate` reliable data pipelines.
- It simplifies `data ingestion, data quality, transformations, dependencies`.


**Q2. Can we use both Python and SQL in DLT?**

- **Yes**. 
- Databricks supports `both Python and SQL` for `declarative pipeline development`.


**Q3. What is the difference between DLT and Auto Loader?**

- Auto Loader is used to incrementally detect and ingest new files.


- **DLT (Delta Live Tables)** is a `Databricks framework` used to `build, manage, and automate` reliable data pipelines.
- It simplifies `data ingestion, data quality, transformations, dependencies`.


**Q4. Does DLT support the Medallion Architecture?**

- **Yes**.
- We can build Bronze, Silver and Gold data layers using streaming tables and materialized views.


**Q5. Is DLT fully managed by Databricks?**

- Databricks manages much of the `pipeline infrastructure, execution and monitoring`, 
- but developers still need to define `source configurations, transformations, data quality rules` and `pipeline settings`.


## 13. Streaming vs Batch Processing – Same Declarative Code?


In Databricks, the `same declarative code` can `support both batch and streaming processing` by changing `how the source data is read`.

- **Batch Processing:**

Processes a `fixed set of available data` at scheduled intervals.

- **Streaming Processing:** 

Processes `incoming data continuously` as new records arrive.


## Batch Processing vs Streaming Processing – Architecture Comparison


### 1. Batch Processing Architecture:


```text
┌──────────────────────────────────────────────────────────────┐
│                      CSV FILES (SOURCE)                       │
│                                                              │
│                 CSV / JSON / Parquet Files                   │
│                                                              │
│       • Receives daily customer order files                  │
│       • Stores orders in different file formats             │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                       BATCH PIPELINE                         │
│                                                              │
│       • Reads available data at scheduled intervals          │
│       • Cleans, filters and transforms records               │
│       • Processes data in large batches                      │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                      DELTA LAKE (SINK)                       │
│                                                              │
│       • Stores processed data                     │
│       • Supports reliable analytical queries                 │
└──────────────────────────────────────────────────────────────┘


### Real-Time Example:


**An e-commerce company processes all customer orders at midnight to generate daily sales reports.**


### 2. Streaming Processing Architecture:


```text
┌──────────────────────────────────────────────────────────────┐
│                       KAFKA (SOURCE)                         │
│                                                              │
│                    Incoming Customer Orders                  │
│                                                              │
│       • Receives incoming data continuously                  │
│       • Publishes order events to Kafka topics               │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                    STREAMING PIPELINE                        │
│                                                              │
│       • Processes incoming records incrementally             │
│       • Filters, cleans and transforms data                  │
│       • Processes data using micro-batches                   │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────┐
│                      DELTA LAKE (SINK)                       │
│                                                              │
│       • Stores processed data                   │
│       • Makes updated data available for analytics           │
└──────────────────────────────────────────────────────────────┘


### Real-Time Example: 


**An e-commerce company processes customer orders as they arrive and updates sales analytics continuously.**


### Architecture Comparison:


| Batch Processing | Streaming Processing |
|---|---|
| CSV Files | Kafka |
| Daily orders | Incoming orders |
| Batch Pipeline | Streaming Pipeline |
| Processes available data | Processes newly arriving data |
| Scheduled execution | Continuous or trigger-based execution |
| Higher processing delay | Lower processing delay |
| Delta Lake | Delta Lake |


### PySpark – Batch Processing:

In [0]:
batch_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/data/orders/")

result = batch_df \
    .groupBy("region") \
    .sum("amount")

display(result)


### PySpark – Streaming Processing:

In [0]:
stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(order_schema) \
    .load("/data/orders/")


### Apply the same transformation:

In [0]:
result = stream_df \
    .groupBy("region") \
    .sum("amount")


### Write streaming results:

In [0]:
result.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "/checkpoints/sales"
    ) \
    .start()


## Important Interview Questions for batch and streaming:


**Q1. Can we use the same transformation logic for batch and streaming?**

- **Yes.** 
- Many **transformations** such as `filter(), select(), withColumn() and dropDuplicates()`, 
- It can be used in both `batch and streaming` applications.


**Q2. What is the main difference between batch and streaming code?**

The main difference is the input reading method:

- **Batch**: spark.read
- **Streaming**: spark.readStream.


**Q3. Can we use GROUP BY in both batch and streaming?**

- **Yes**. 
- However, streaming aggregations require `state management` and `appropriate output modes, triggers` and `sometimes watermarks`.


**Q4. Does streaming always mean continuous execution?**

- **No**,
- **Structured Streaming** supports both `continuous execution patterns` and `trigger-based incremental processing`, such as **AvailableNow**.


**Q5. Which is better: Batch or Streaming?**

**Neither** is better.

- **Batch** is better when `immediate results` are unnecessary.
- **Streaming** is better when `businesses need fresh data` with low latency.